# Faster R-CNN results: ignore-aware baseline on every fifth frame

We compare the saved measurements of the Faster R-CNN experiments using tables and Plotly charts.
All runs use seed 2026, batch size 2, resize 640/1024, SGD with learning rate 0.0025, mixed precision
on CUDA and five epochs on the same 4,868 training frames as the RetinaNet runs. The reported model
is the final epoch, and validation values per epoch are information only. AP values use the 0 to 1
scale, and every table shows COCO AP at maxDets 100, which the RetinaNet results use, and at maxDets
500, the number of detections the submission scores. Missing records show as **Not run** and
partial records as **Partial**.

In [ ]:
import json
import sys
from pathlib import Path

import plotly.express as px
import polars as pl
from decouple import config
from IPython.display import Markdown, display

ROOT = Path.cwd() if Path("pyproject.toml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from hm2.faster_rcnn_experiments import aggregate_loss_curve, experiment_label, load_batch_records

BATCH_ID = "faster_rcnn_stride5_5epochs_seed2026"
SAVED_RESULTS = ROOT / "runs/faster_rcnn" / BATCH_ID
batch_dir = Path(config("FASTER_RCNN_BATCH_DIR", default=str(SAVED_RESULTS))).expanduser()
records = (
    load_batch_records(batch_dir) if batch_dir.is_dir() else {"experiments": [], "probe": None}
)
experiments = [item for item in records["experiments"] if item["status"] != "Not run"]
pl.Config.set_tbl_cols(12)
pl.Config.set_tbl_rows(32)
pl.Config.set_fmt_str_lengths(90)
if not experiments:
    display(
        Markdown(
            "**Not run.** No experiment records were found. Download the output folder "
            "`runs/faster_rcnn/<batch id>/` of the Kaggle run into the repository root. "
            "See the instructions at the end of this notebook."
        )
    )

## 1. Timing probe

We time a few real updates and the inference speed before the full run, so that we know the cost
of an epoch. The projections extrapolate linearly from a few updates of an untrained detector and
are not results.

In [ ]:
probe = records["probe"]
if probe is None:
    display(Markdown("**Not run.** The timing probe has no record."))
else:
    display(
        pl.DataFrame(
            {
                "Measurement": [
                    "Device",
                    "Training images per second",
                    "Seconds per update",
                    "Peak GPU MiB",
                    "Inference ms per frame, batch 1",
                    "Projected training minutes",
                    "Projected validation minutes",
                ],
                "Value": [
                    probe["gpu"] or probe["device"],
                    f"{probe['train_images_per_second']:.2f}",
                    f"{probe['train_seconds_per_update']:.2f}",
                    f"{probe['peak_gpu_mib']:.0f}" if probe["peak_gpu_mib"] else "None",
                    f"{probe['inference']['batch_1']['milliseconds_per_frame']:.1f}",
                    f"{probe['projection']['train_minutes']:.0f}",
                    f"{probe['projection']['validation_minutes']:.0f}",
                ],
            }
        )
    )

## 2. Completed training budget

The training selection contains 4,868 frames from all 56 training clips, the same frames as the
RetinaNet stride-five runs, and validation uses the 2,846 official validation frames from seven
separate clips with no shared flights. Training time is the sum of end-to-end batch latencies; it
excludes setup, validation, checkpoints and time between sessions.

In [ ]:
if experiments:
    display(
        pl.from_dicts(
            [
                {
                    "Experiment": experiment_label(item["experiment_id"]),
                    "Status": item["status"],
                    "Train frames": item["train_frames"],
                    "Epochs": f"{item['epochs_completed']} of {item['epochs_total']}",
                    "Optimizer updates": item["global_step"],
                    "Image exposures": item["image_exposures"],
                    "Training batch time (min)": round(item["wall_train_seconds"] / 60, 2),
                    "Training images/s": round(
                        item["image_exposures"] / item["wall_train_seconds"], 2
                    ),
                }
                for item in experiments
            ]
        )
    )

## 3. Validation at the last completed epoch

We compare the models under the same validation protocol. The inference speed on 300 validation
frames at batch size 1 includes decoding, collation, transfers, model transforms, inference, NMS and
CPU outputs, and excludes COCO evaluation. It comes from one session and is not a controlled
hardware benchmark.

In [ ]:
if experiments:
    display(
        pl.from_dicts(
            [
                {
                    "Experiment": experiment_label(item["experiment_id"]),
                    "Epoch": item["epochs"][-1]["epoch"],
                    "mAP, maxDets 100": item["epochs"][-1]["map_100"],
                    "AP50, maxDets 100": item["epochs"][-1]["ap50_100"],
                    "AP small, maxDets 100": item["epochs"][-1]["ap_small_100"],
                    "mAP, maxDets 500": item["epochs"][-1]["map_500"],
                    "AP50, maxDets 500": item["epochs"][-1]["ap50_500"],
                    "AP small, maxDets 500": item["epochs"][-1]["ap_small_500"],
                    "Validation ms/frame, batch 2": item["epochs"][-1]["validation_ms_per_frame"],
                    "Inference ms/frame, batch 1": (
                        item["speed"]["milliseconds_per_frame"] if item["speed"] else None
                    ),
                }
                for item in experiments
                if item["epochs"]
            ],
            infer_schema_length=None,
        )
    )

## 4. Validation by epoch

We list every completed validation. These values show how the metrics move during training and
do not select a checkpoint.

In [ ]:
epoch_rows = [
    {"Experiment": experiment_label(item["experiment_id"]), **row}
    for item in experiments
    for row in item["epochs"]
]
if epoch_rows:
    display(
        pl.from_dicts(epoch_rows, infer_schema_length=None).select(
            "Experiment",
            pl.col("epoch").alias("Epoch"),
            pl.col("map_100").alias("mAP, maxDets 100"),
            pl.col("ap50_100").alias("AP50, maxDets 100"),
            pl.col("ap_small_100").alias("AP small, maxDets 100"),
            pl.col("map_500").alias("mAP, maxDets 500"),
            pl.col("ap_small_500").alias("AP small, maxDets 500"),
            pl.col("evaluation_seconds").round(1).alias("Validation seconds"),
        )
    )

## 5. Class AP at the last completed epoch

We retain the eight assignment categories and report the same COCO AP definition for each class
at both detection caps.

In [ ]:
class_rows = [
    {
        "Class": class_name,
        "Experiment": f"{experiment_label(item['experiment_id'])}, maxDets {cap}",
        "AP": score,
    }
    for item in experiments
    if item["epochs"]
    for cap in (100, 500)
    for class_name, score in item[f"per_class_ap_{cap}"].items()
]
if class_rows:
    display(pl.from_dicts(class_rows).pivot(on="Experiment", index="Class", values="AP"))

## 6. Validation and training curves

The validation curves contain every completed validation. The loss curves show batch means over
windows of at most 100 attempts, split at epoch boundaries, and the horizontal axis counts
successful optimizer updates.

In [ ]:
if epoch_rows:
    validation_curves = (
        pl.from_dicts(epoch_rows, infer_schema_length=None)
        .unpivot(
            index=["Experiment", "epoch"],
            on=["map_100", "map_500", "ap_small_100", "ap_small_500"],
            variable_name="Metric",
            value_name="AP",
        )
        .with_columns(
            pl.col("Metric").replace_strict(
                {
                    "map_100": "mAP, maxDets 100",
                    "map_500": "mAP, maxDets 500",
                    "ap_small_100": "AP small, maxDets 100",
                    "ap_small_500": "AP small, maxDets 500",
                }
            )
        )
    )
    px.line(
        validation_curves,
        x="epoch",
        y="AP",
        color="Metric",
        markers=True,
        title="Validation AP by epoch",
        labels={"epoch": "Epoch", "AP": "Average precision"},
    ).show(renderer="plotly_mimetype", config={"responsive": False})

In [ ]:
loss_names = {
    "loss": "Total",
    "loss_objectness": "RPN objectness",
    "loss_rpn_box_reg": "RPN box regression",
    "loss_classifier": "Box classification",
    "loss_box_reg": "Box regression",
}
for item in experiments:
    history_path = batch_dir / item["experiment_id"] / "history.json"
    if not history_path.is_file():
        continue
    curve = aggregate_loss_curve(json.loads(history_path.read_text())).unpivot(
        index=["epoch", "step"],
        on=list(loss_names),
        variable_name="Loss",
        value_name="Mean loss",
    )
    px.line(
        curve.with_columns(pl.col("Loss").replace_strict(loss_names)),
        x="step",
        y="Mean loss",
        color="Loss",
        title=f"Training loss: {experiment_label(item['experiment_id'])}",
        labels={"step": "Optimizer update"},
    ).show(renderer="plotly_mimetype", config={"responsive": False})

## 7. Evaluation boundaries and pending comparison

We use the original category-specific COCO crowd annotations, COCO IoU thresholds 0.50 to 0.95 and
maxDets 100 and 500. Crowd handling follows COCO semantics and does not establish equivalence to
the competition's metric or its ignore semantics, which remain unverified. Training excludes
background samples that lie mostly inside ignored regions, so this run and the RetinaNet runs differ
in that handling as well as in the model. The comparison with the RetinaNet results needs both sets
of records under the same protocol, and a run with ignore handling switched off separates the two
effects. CUDA kernels and runtime changes can affect numerical reproducibility, and each result is
a single seed.

## 8. Reproducing this comparison

We keep the records of each run in the Kaggle output folder `runs/faster_rcnn/<batch id>/`: the
configuration, environment, per-epoch metrics, loss history, timing and the test file, with the
checkpoint `latest.pt` next to them. To rerun this notebook, download that folder from the Kaggle
notebook output and place it at `runs/faster_rcnn/<batch id>/` in the repository root, or set
`FASTER_RCNN_BATCH_DIR` to its location. The records are gitignored input and can be deleted after
viewing. Rerunning regenerates the presentation of the saved measurements, not the training or the
COCO evaluation.